# Processed E-commerce Dataset — Day 9

**Objective:** Load the Orders, Customers, and Products CSV files, combine related information with `merge()`, demonstrate `concat()`, use `apply()` for a useful transformation, perform DateTime operations, and export a clean processed dataset as CSV.

## 1. Import Libraries

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Load the Datasets

In [ ]:
customers = pd.read_csv("data/Customers.csv")
orders = pd.read_csv("data/Orders.csv")
products = pd.read_csv("data/Products.csv")

print("Customers shape:", customers.shape)
print("Orders shape:", orders.shape)
print("Products shape:", products.shape)

## 3. Inspect the Datasets

In [ ]:
print("Customers:")
display(customers.head())

print("Orders:")
display(orders.head())

print("Products:")
display(products.head())

## 4. Basic Data Quality Checks

In [ ]:
print("Missing values:")
print("Customers:\n", customers.isna().sum())
print("\nOrders:\n", orders.isna().sum())
print("\nProducts:\n", products.isna().sum())

print("\nDuplicate rows:")
print("Customers:", customers.duplicated().sum())
print("Orders:", orders.duplicated().sum())
print("Products:", products.duplicated().sum())

## 5. Convert the Order Date to DateTime

The `Order_Date` column is converted from text to Pandas DateTime so that date-based information can be extracted easily.

In [ ]:
orders["Order_Date"] = pd.to_datetime(orders["Order_Date"])

print(orders["Order_Date"].dtype)
display(orders[["Order_ID", "Order_Date"]].head())

## 6. Demonstrate `concat()`

Here the Orders DataFrame is split into two parts and then recombined vertically using `pd.concat()`. This demonstrates combining DataFrames with the same column structure.

In [ ]:
orders_part1 = orders.iloc[:len(orders)//2].copy()
orders_part2 = orders.iloc[len(orders)//2:].copy()

orders_combined = pd.concat([orders_part1, orders_part2], ignore_index=True)

print("First part:", orders_part1.shape)
print("Second part:", orders_part2.shape)
print("After concat:", orders_combined.shape)

display(orders_combined.head())

## 7. Merge Orders with Customer Information

`merge()` is used with `Customer_ID` to attach customer details to every order.

In [ ]:
orders_customers = orders_combined.merge(
    customers,
    on="Customer_ID",
    how="left"
)

print("Shape after customer merge:", orders_customers.shape)
display(orders_customers.head())

## 8. Merge with Product Information

The result is merged with the Products DataFrame using `Product_ID` to add product name, category, price, and brand.

In [ ]:
processed = orders_customers.merge(
    products,
    on="Product_ID",
    how="left"
)

print("Shape after product merge:", processed.shape)
display(processed.head())

## 9. Create Useful Columns Using `apply()`

`Order_Value` is calculated from quantity and unit price. Then `apply()` with a lambda function classifies each order into a value category.

In [ ]:
processed["Order_Value"] = processed["Quantity"] * processed["Unit_Price"]

processed["Order_Value_Category"] = processed["Order_Value"].apply(
    lambda x: "High Value" if x >= 5000
    else ("Medium Value" if x >= 2000 else "Low Value")
)

display(
    processed[
        ["Order_ID", "Quantity", "Unit_Price", "Order_Value", "Order_Value_Category"]
    ].head(10)
)

## 10. DateTime Operations

In [ ]:
processed["Order_Year"] = processed["Order_Date"].dt.year
processed["Order_Month"] = processed["Order_Date"].dt.month
processed["Order_Month_Name"] = processed["Order_Date"].dt.strftime("%B")
processed["Order_Day"] = processed["Order_Date"].dt.day
processed["Order_Day_of_Week"] = processed["Order_Date"].dt.day_name()

display(
    processed[
        [
            "Order_Date", "Order_Year", "Order_Month",
            "Order_Month_Name", "Order_Day", "Order_Day_of_Week"
        ]
    ].head(10)
)

## 11. Organize the Final Clean DataFrame

In [ ]:
column_order = [
    "Order_ID", "Order_Date", "Order_Year", "Order_Month", "Order_Month_Name",
    "Order_Day", "Order_Day_of_Week", "Customer_ID", "Customer_Name", "City",
    "Region", "Membership_Type", "Product_ID", "Product_Name", "Category",
    "Brand", "Unit_Price", "Quantity", "Order_Value", "Order_Value_Category",
    "Payment_Method", "Order_Status"
]

processed = processed[column_order].sort_values("Order_Date").reset_index(drop=True)

print("Final shape:", processed.shape)
display(processed.head(10))

## 12. Final Dataset Validation

In [ ]:
print("Missing values in final dataset:")
display(processed.isna().sum())

print("Final number of rows:", len(processed))
print("Final number of columns:", len(processed.columns))
print("Date range:", processed["Order_Date"].min().date(), "to", processed["Order_Date"].max().date())

## 13. Export the Processed Dataset

In [ ]:
output_path = "data/processed_ecommerce_dataset.csv"
processed.to_csv(output_path, index=False)

print(f"Processed dataset exported successfully to: {output_path}")

## 14. Conclusion

The three e-commerce datasets were successfully loaded and processed using Pandas. `concat()` was demonstrated by recombining order DataFrames, while `merge()` combined order data with customer and product information. `apply()` was used to classify order values, and DateTime operations extracted year, month, day, month name, and day of the week. The final clean dataset was exported as `processed_ecommerce_dataset.csv`.